# 主幹判別式微調（humanv0）

執行 `aicup.trunk_cls`：humanv0 的最後幾個 block 加一個線性頭，做單局 10 級分類，輸出單局 log 機率給競賽二的融合。

- 這份 notebook 是實驗，不在競賽二最終版本流程內：newdev 的 log loss 沒有通過門檻。
- 外部資源：KataGo `python/`（tag `v1.15.0`，MIT）與 `b18c384nbt-humanv0.ckpt`。checkpoint 先找 Drive 快取，沒有才下載
- 棋譜由本機的 `aicup.trunk_games` 先切好，Colab 不需要完整訓練集
- 輸出在 Drive 的 `rank-trained-models/trunk-cls/`；`train` 到 `TIME_LIMIT` 小時存 `last.pt` 停止，重跑同一個 MODE 接續，接續時 `COMMIT` 要和開始時相同

| MODE | 內容 |
|---|---|
| `smoke` | 200 位訓練玩家、20 步，輸出到 `trunk-cls-smoke/` |
| `dtype` | 比較 fp32 與 fp16、bf16 的 logit；用 fp16 訓練時 policy 那一行要通過：最大絕對差 ≤ 0.05、argmax 一致率 ≥ 99% |
| `train` | cap 20、3 個 epoch，每個 epoch 結束印 newdev 各級等權 log loss |
| `infer` | 用 `EPOCH` 的權重推論 val648 題目與測試題，`EPOCH` 填 newdev log loss 最低的 epoch |

## 需要的檔案

路徑都在 Drive 的 `MyDrive/aicup-2026/` 下，在本機產生後上傳。

| 路徑 | MODE | 產生方式（本機，repo 根目錄） |
|---|---|---|
| `humansl-analysis/questions/questions-s42.npz` | 全部 | `python -m aicup.export_questions --seeds 42`，輸入是 `python -m aicup.hsl_games --run val648` 的 games.csv |
| `humansl-analysis/trunk-games/{train,newdev,question}-games.csv` | 全部 | `python -m aicup.trunk_games --questions humansl-analysis/questions/questions-s42.npz` |
| `tests/rank_prediction_test_{public,private}.csv` | `infer` | AI CUP 競賽平台 |

## 參數

In [ ]:
REPO_URL = 'https://github.com/Jx-study/aicup-2026-go.git'
COMMIT = 'origin/main'  # 重現特定版本時填 commit hash
MODE = 'smoke'
TIME_LIMIT = 4.5  # 小時，train 到時存 last.pt 停止
EPOCH = None  # infer 用的 epoch，填 newdev log loss 最低者
AUTO_UNASSIGN = True

GAMES = 'humansl-analysis/trunk-games/'
QUESTIONS = 'humansl-analysis/questions/questions-s42.npz'
HUMANV0 = '/content/models/b18c384nbt-humanv0.ckpt'
DATA = ['--train-csv', GAMES + 'train-games.csv', '--newdev-games-csv', GAMES + 'newdev-games.csv']
BASE = ['python', '-m', 'aicup.trunk_cls']
if MODE not in ('smoke', 'dtype', 'train', 'infer'):
    raise SystemExit('MODE 要是 smoke、dtype、train 或 infer')
OUT = 'rank-trained-models/trunk-cls' + ('-smoke' if MODE == 'smoke' else '')
NEEDS = [(QUESTIONS, 'python -m aicup.export_questions --seeds 42')]
NEEDS += [(GAMES + n, 'python -m aicup.trunk_games --questions ' + QUESTIONS)
          for n in ('train-games.csv', 'newdev-games.csv', 'question-games.csv')]
if MODE == 'infer':
    NEEDS += [('tests/rank_prediction_test_%s.csv' % s, 'AI CUP 競賽平台') for s in ('public', 'private')]


def commands(train_dtype, workers):
    """MODE 對應的指令清單；train_dtype、workers 在取得 GPU 型號與核心數後才決定。"""
    common = ['--checkpoint', HUMANV0, '--infer-dtype', 'fp32', '--workers', str(workers)]
    train = [*BASE, 'train', *DATA, '--out', OUT, '--train-dtype', train_dtype, *common, '--cap', '20', '--epochs', '3']
    if MODE == 'smoke':
        return [[*train, '--max-steps', '20', '--max-train-players', '200']]
    if MODE == 'dtype':
        return [[*BASE, 'dtype', *DATA, '--checkpoint', HUMANV0, '--games', '8']]
    if MODE == 'train':
        return [[*train, '--time-limit', str(TIME_LIMIT)]]
    if EPOCH is None:
        raise SystemExit('infer 要先填 EPOCH')
    ckpt = ['--ckpt', '%s/epoch-%d.pt' % (OUT, EPOCH)]
    return [[*BASE, 'infer', *ckpt, '--questions', QUESTIONS, '--games-csv', GAMES + 'question-games.csv',
             '--out', '%s/val-games-epoch-%d.npz' % (OUT, EPOCH), *common],
            [*BASE, 'infer', *ckpt, '--test', 'tests', '--out', '%s/test-games-epoch-%d.npz' % (OUT, EPOCH), *common]]


for c in commands('<依 GPU>', '<核心數>'):
    print(' '.join(c))

## 取得程式

要看到：Drive 掛載完成、最新的 commit、GPU 型號。

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sgfmill'], check=True)
from aicup.colab_nb import DRIVE_ROOT, check_commit, link_drive, need, run

GIT_HEAD = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()

!git log --oneline -1
!nvidia-smi -L

## 準備資料

要看到：需要的檔案都在、KataGo `python/` 的 commit 是 v1.15.0、humanv0 通過大小與 SHA-256 檢查、訓練精度。

In [ ]:
import shutil

link_drive(['humansl-analysis', 'rank-trained-models', 'tests'])
need(NEEDS)
if MODE in ('smoke', 'train'):
    check_commit(OUT, GIT_HEAD)
KATAGO = '/content/KataGo'
KATAGO_COMMIT = 'c560d38585b446bc93b0a62c77364bc8e7d266a4'  # tag v1.15.0
if not os.path.exists(KATAGO + '/python/features.py'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.15.0', '--filter=blob:none', '--sparse',
                    'https://github.com/lightvector/KataGo', KATAGO], check=True)
    subprocess.run(['git', '-C', KATAGO, 'sparse-checkout', 'set', 'python'], check=True)
head = subprocess.run(['git', '-C', KATAGO, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert head == KATAGO_COMMIT, 'KataGo 的 commit 不是 v1.15.0'
os.environ['KATAGO_PYTHON'] = KATAGO + '/python'

# katagotraining.org 會擋 urllib，先用 Drive 快取，沒有才用 wget
CACHE = DRIVE_ROOT + '/katago/b18c384nbt-humanv0.ckpt'
os.makedirs(os.path.dirname(HUMANV0), exist_ok=True)
if not os.path.exists(HUMANV0):
    if os.path.exists(CACHE):
        shutil.copy(CACHE, HUMANV0)
    else:
        # 先下載到暫存檔，中斷時不留下不完整的 checkpoint
        subprocess.run(['wget', '-q', '-O', HUMANV0 + '.part',
                        'https://media.katagotraining.org/uploaded/networks/models_extra/b18c384nbt-humanv0.ckpt'], check=True)
        os.replace(HUMANV0 + '.part', HUMANV0)
from aicup import katago_torch as kt

kt.ensure_humanv0(HUMANV0)  # 檢查大小與 SHA-256
if not os.path.exists(CACHE):
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    shutil.copy(HUMANV0, CACHE)

import torch

major, _ = torch.cuda.get_device_capability()
TRAIN_DTYPE = 'bf16' if major >= 8 else 'fp16'  # Ampere 之後才有原生 bf16
CMDS = commands(TRAIN_DTYPE, os.cpu_count())
print('訓練精度', TRAIN_DTYPE, '，CPU 核心', os.cpu_count())

## 執行

輸出同時寫到 Drive 的 `<輸出目錄>/<MODE>.log`。

In [ ]:
for cmd in CMDS:
    run(cmd, os.path.join(OUT, MODE + '.log'))

## 檢查

要看到：每個 epoch 的 newdev 各級等權 log loss 與正確率，以及輸出目錄的檔案。

In [ ]:
import json

path = os.path.join(OUT, 'history.jsonl')
if os.path.exists(path):
    for line in open(path):
        r = json.loads(line)
        if 'newdev_nll' in r:
            print('epoch %d newdev log loss %.4f 正確率 %.4f' % (r['epoch'], r['newdev_nll'], r['newdev_acc']))
print(sorted(os.listdir(OUT)) if os.path.isdir(OUT) else '沒有輸出目錄 %s' % OUT)

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣運算單元。

In [ ]:
from google.colab import runtime

drive.flush_and_unmount()  # 等 Drive 寫入完成
if AUTO_UNASSIGN:
    runtime.unassign()
else:
    print('未釋放 runtime，用完記得手動中斷')